# Task 3 — Validate

**Purpose:** final task. Checks the output of Task 2 is sane (row count > 0, no nulls in key columns). Runs after `transform_data` succeeds.

**Concept demoed:** Retry logic. Flip `force_fail` to `true` in the Job parameters before a demo run — this task will raise an exception, and you can show the whole team the Job UI automatically re-running just this one task (not the whole pipeline) according to the retry policy you set.

In [ ]:
dbutils.widgets.text("catalog", "main", "Target catalog")
dbutils.widgets.text("schema", "default", "Target schema")
dbutils.widgets.dropdown("force_fail", "false", ["false", "true"], "Force a failure (demo retries)")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
force_fail = dbutils.widgets.get("force_fail") == "true"

try:
    summary_table = dbutils.jobs.taskValues.get(
        taskKey="transform_data", key="summary_table",
        default=f"{catalog}.{schema}.demo_trips_summary"
    )
except Exception:
    summary_table = f"{catalog}.{schema}.demo_trips_summary"

print(f"Validating: {summary_table}")
print(f"force_fail = {force_fail}")

In [ ]:
import time

# Purely for the live demo: makes the task fail deterministically so the
# team can watch the Job UI retry it. In real pipelines this block wouldn't
# exist - only the actual validation checks below would.
if force_fail:
    print("Simulating a transient failure (e.g. flaky upstream source)...")
    time.sleep(2)
    raise Exception("DEMO: Forced failure to illustrate retry behavior. "
                     "Set force_fail=false to let this task pass.")

In [ ]:
df = spark.table(summary_table)
row_count = df.count()
null_zip_count = df.filter(df.pickup_zip.isNull()).count()

print(f"Row count: {row_count}")
print(f"Null pickup_zip rows: {null_zip_count}")

assert row_count > 0, "Validation failed: summary table is empty"
assert null_zip_count == 0, "Validation failed: found null pickup_zip values"

print("Validation passed.")

**Talking point — Notifications:** this task doesn't need any notification code at all. In the Job UI, under this task's (or the whole Job's) **Notifications** settings, you add an email/Slack/webhook destination and choose triggers: *on start*, *on success*, *on failure*, *on duration threshold exceeded*. Databricks sends the alert automatically - the notebook stays clean.

**Talking point — External Locations / Storage Credentials (concept only on Free Edition):** if this pipeline needed to read raw files from our own GCS bucket instead of `samples.nyctaxi.trips`, we'd register a **Storage Credential** (the GCP service account Databricks assumes) and an **External Location** (the specific `gs://bucket/path` that credential is scoped to) in Unity Catalog under *Catalog > External Data*. Free Edition blocks this screen, so today it's a walkthrough of the screen only, not a live click-through.